# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# ETL Job One: Facility Hours by Month - Parquet Export

# Extract: Read from managed tables
bookings_df = spark.table("workspace.default.bookings")
facilities_df = spark.table("workspace.default.facilities")

# Transform: Calculate facility hours by month
# Join bookings with facilities and calculate hours (slots * 0.5 since each slot is 30 min)
from pyspark.sql.functions import col, month, year, sum as _sum, concat_ws

facility_hours_by_month = (
    bookings_df
    .join(facilities_df, "facid")
    .withColumn("month", month(col("starttime")))
    .withColumn("year", year(col("starttime")))
    .withColumn("hours", col("slots") * 0.5)
    .groupBy("facid", "name", "year", "month")
    .agg(_sum("hours").alias("total_hours"))
    .orderBy("year", "month", "facid")
)

display(facility_hours_by_month)

# Load: Save to parquet file
output_path = "/Volumes/workspace/default/etl_output/facility_hours_by_month.parquet"
facility_hours_by_month.write.mode("overwri/Volumes/workspace/default/etl_output/facility_hours_by_month.parquet/_committed_2821631140856035786te").parquet(output_path)

print(f"Data successfully saved to {output_path}")

facid,name,year,month,total_hours
0,Tennis Court 1,2012,7,135.0
1,Tennis Court 2,2012,7,103.5
2,Badminton Court,2012,7,90.0
3,Table Tennis,2012,7,52.0
4,Massage Room 1,2012,7,132.0
5,Massage Room 2,2012,7,12.0
6,Squash Court,2012,7,82.0
7,Snooker Table,2012,7,78.0
8,Pool Table,2012,7,58.5
0,Tennis Court 1,2012,8,229.5


Data successfully saved to /Volumes/workspace/default/etl_output/facility_hours_by_month.parquet


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# ETL Job Two: Three-Way Join with Partitioning

# Extract: Read from managed tables
bookings_df = spark.table("workspace.default.bookings")
facilities_df = spark.table("workspace.default.facilities")
members_df = spark.table("workspace.default.members")

# Transform: Join all three tables (threejoin exercise)
# Join bookings with members and facilities to get complete booking information
from pyspark.sql.functions import concat_ws

threejoin_result = (
    bookings_df
    .join(members_df, bookings_df.memid == members_df.memid, "inner")
    .join(facilities_df, bookings_df.facid == facilities_df.facid, "inner")
    .select(
        bookings_df.bookid,
        facilities_df.name.alias("facility"),
        concat_ws(" ", members_df.firstname, members_df.surname).alias("member_name"),
        bookings_df.starttime,
        bookings_df.slots,
        facilities_df.facid
    )
    .orderBy("starttime")
)

display(threejoin_result)

# Load: Partition by facility and save to managed table
# Using Delta format (Databricks default for managed tables)
threejoin_result.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("facility") \
    .saveAsTable("workspace.default.threejoin_delta")

print("Data successfully saved to threejoin_delta table, partitioned by facility")

bookid,facility,member_name,starttime,slots,facid
1,Massage Room 1,Darren Smith,2012-07-03T08:00:00.000Z,2,4
4,Pool Table,Darren Smith,2012-07-03T10:00:00.000Z,1,8
0,Table Tennis,Darren Smith,2012-07-03T11:00:00.000Z,2,3
5,Pool Table,Darren Smith,2012-07-03T15:00:00.000Z,1,8
2,Squash Court,GUEST GUEST,2012-07-03T18:00:00.000Z,2,6
3,Snooker Table,Darren Smith,2012-07-03T19:00:00.000Z,2,7
6,Tennis Court 1,Tracy Smith,2012-07-04T09:00:00.000Z,3,0
15,Pool Table,Tracy Smith,2012-07-04T12:00:00.000Z,1,8
11,Squash Court,GUEST GUEST,2012-07-04T12:30:00.000Z,2,6
8,Massage Room 1,Tim Rownam,2012-07-04T13:30:00.000Z,2,4


Data successfully saved to threejoin_delta table, partitioned by facility


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# ETL Job Three: Extract Stock Data via HTTP Requests

import requests
from pyspark.sql.functions import col, weekofyear, year, max as _max, lit
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

API_KEY = "8c0dc9654bmshbd8545b4874e8d7p17ef4cjsnc8f5bdd41da2"  

# Extract: Fetch stock data from Alpha Vantage API
companies = {
    "GOOG": "Google",
    "AAPL": "Apple", 
    "MSFT": "Microsoft",
    "TSLA": "Tesla"
}

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": API_KEY
}

# Schema for stock data
schema = StructType([
    StructField("date", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("company", StringType(), True)
])

all_stock_data = None

# Loop through each company to fetch data
for symbol, company_name in companies.items():
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }
    
    try:
        response = requests.get(url, headers=headers, params=querystring)
        data = response.json()
        
        # Extract time series data
        time_series = data.get("Time Series (Daily)", {})
        
        # Convert to list of rows
        rows = []
        for date, values in time_series.items():
            rows.append((
                date,
                float(values.get("1. open", 0)),
                float(values.get("2. high", 0)),
                float(values.get("3. low", 0)),
                float(values.get("4. close", 0)),
                float(values.get("5. volume", 0)),
                company_name
            ))
        
        # Create DataFrame
        company_df = spark.createDataFrame(rows, schema)
        
        # Union with existing data
        if all_stock_data is None:
            all_stock_data = company_df
        else:
            all_stock_data = all_stock_data.union(company_df)
            
        print(f"Fetched {len(rows)} records for {company_name}")
        
    except Exception as e:
        print(f"Error fetching data for {symbol}: {str(e)}")

# Transform: Calculate weekly max closing price
if all_stock_data is not None:
    # Convert date string to date type and extract week/year
    weekly_max_closing = (
        all_stock_data
        .withColumn("date", col("date").cast(DateType()))
        .withColumn("week", weekofyear(col("date")))
        .withColumn("year", year(col("date")))
        .groupBy("company", "year", "week")
        .agg(_max("close").alias("max_closing_price"))
        .orderBy("company", "year", "week")
    )
    
    display(weekly_max_closing)
    
    # Load: Partition by company and save to managed table
    weekly_max_closing.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("company") \
        .saveAsTable("workspace.default.max_closing_price_weekly")
    
    print("Data successfully saved to max_closing_price_weekly table, partitioned by company")
else:
    print("No data fetched. Please check your API key and connection.")

Fetched 100 records for Google
Fetched 100 records for Apple
Fetched 100 records for Microsoft
Fetched 100 records for Tesla


company,year,week,max_closing_price
Apple,2025,1,273.76
Apple,2025,47,271.49
Apple,2025,48,278.85
Apple,2025,49,286.19
Apple,2025,50,278.78
Apple,2025,51,274.61
Apple,2025,52,273.81
Apple,2026,1,271.01
Apple,2026,2,267.26
Apple,2026,3,261.05


Data successfully saved to max_closing_price_weekly table, partitioned by company


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# ETL Job Four: Extract from PostgreSQL RDBMS

# Extract: Connect to RNAcentral public PostgreSQL database
# Database connection details from: https://rnacentral.org/help/public-database

jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# Read 100 records from the rna table
query = "(SELECT * FROM rna LIMIT 100) AS rna_sample"

try:
    rna_df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )
    
    print(f"Successfully extracted {rna_df.count()} RNA records")
    display(rna_df)
    
    # Load: Save to managed table (no transformation required)
    rna_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("workspace.default.rna_100_records")
    
    print("Data successfully saved to rna_100_records table")
    
except Exception as e:
    print(f"Error connecting to PostgreSQL database: {str(e)}")
    print("Note: Ensure PostgreSQL JDBC driver is available on the cluster")

Successfully extracted 100 RNA records


id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
3503296,URS00003574C0,2014-05-29T13:51:05.000Z,RNACEN,418DD8E66852C307,1746,CATGCATGTCTCAGTACACGCCCCTGTACGGTGAAACTGCGAACGGCTCATTACATCAGTTATCGTGTCCTCGATCGTTCCTTACCACATGGATACCCGTGGTAATTCTAGAGCTAATACATGCGCCCAGTCCCGACCGCTCGCCGGAAGGGATGTATTTATTAGATTTTCAGACCATGACGGGGCAACCCGTGCGTTGGTGATTCATGGTAACTGCTCGGATCGCACGGCCTCGCGCCGGCGACGCCTCGTTCCAATTTCTGCCCTATCAACTGACGATGGTACGGTAGTGGCCTACCATGGTCGTAACGGGTGACGGAGAATCAGGGTTCGGTTCCGGAGAGGGAGCCCGAGAAATGGCTACCACTTCCACGGAAGGCAGCAGGCGCACACATTGCCCAATCCCGACACGGGGAGGCAGTGACAAGAAATAACGGTGCGTCGGCTTAGGCCGTCGCAACCGGAATGAGTACGACCCAAATCCTCTAACGAGGATCCATTGGAGGGCAAGTCTGGTGCCAGCAGCCGCGGTAATTCCAGCTCCAACAGTGTATGCTAATGTTGCTGCAGTTAAAAAGCTCGTAGTTGGATCGCAGGCGGTCGACGTCGAGGTGCGCTACACGCGCTACCTCGCGTCGTCGCCTTATCCGCGGAGGCCGCGCGGTGCTCTTGATCGAGCGCCGGCCGGTCTACCCGGGACGTCACCTTGAGGAAACTAGAGTGTTCAAGGCAGGTGTCTCGCCCTGAATACCGCAGCATGGAATGACACGCGGGACTGTGCGACGCGCAAGCGTCTCGGTCCGGTGCGCGCGTTGGTCTGACGGACCGCAGTGACGGTTAAGAGGGACGGTCGGGGGCATTCGTATTTCGTCGTGAGAGGTGAAATTCTTAGACCGACGAAAGACGGACAAGTGCGAAAGCGTTTGCCAAGAACGTTTTCATTAATCAAGAACGAAAGTTAGAGGATCGAAGACGATCAGATACCGTCGTAGTTCTAACCATAAACGATGCCGACCAGGGATCGGCGGGCCGTGTTCATCGAGCTCGTCGGCACCTCCCGGGAAACCTGAGTGTTTGGGTTCCGGGGGGAGTATGGTCGCAAGACTGAAACTTAAAGGAATTGACGGAAGGGCACCACCAGGAGTGGAGCCTGCGGCTTAATTTGACTCAACACGGGGAACCTCACCAGGTCCGGACATGACGAGGATTGACAGACTGACCGCTCTTTCTCGATCTCATGGTTGGTGGTGCATGGCCGTTCTTAGTTGGTGGAGCGATTTGTCAGGTTAATTCCGATAACGAACGAGACCTTGTCCGGCTAGTTGAGCGCGCGATCCTCGGATCGCGTCCGCTATCTAGAGGCACTGTCGGTGTGCGCAAGCCGAAGCGAGGAAGGCAATAACAGGTCTGTGATGCCCTTCGATGTTCTGGGCCGCACGCGCGCTACAATGACGAGGTCAGCGAGTCCGCCTCCACCGGAAGGTGTCGGGTAATCTTGCGAAACGTYGTCGTGATGGGGACAGAGCATTGAAATTATTGCTCTCGAACGAGGAATTCCTAGTAGGCGCGAGTCATCAGCTCGCGTCGATTACGTCCCTGCCCTTTGTACACACCGCCCGTCGCTACTACCGATTGGATGGTTTAGTGAGAGCTCGGGACCGGCCCGGGCGGCCCCCTCKNGGGGGCCGCCCCGCGGTCGGGAACCTGGTCGAACTTGATCATTTAGAGGAAGTAAAAGTCGTAACA,null,cbb79cf108ab31e0235b8fe107d9e07c
3503298,URS00003574C2,2014-05-29T13:51:05.000Z,RNACEN,F6006F496E6346CF,29,GATAGTAGGATGGAATTTTATTCAAAGCT,null,cbb7a60e8f67a48e91e2e2944b60088d
3503299,URS00003574C3,2014-05-29T13:51:05.000Z,RNACEN,188068EB59C61A20,1391,AGAGTTTGCTCCTGGCTCAGGATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGAAGCACTTGAGATTGATTCTTCGGAAGATTTCTCTTGTGACTTAGTGGCGGACGGGTGAGTAACGCGTGGGTAACCTGCCTCACACAGGGGGATAACAGTTGGAAACGACCGCTAATACCGCATAACCCGCTAGGGCCGCATGGCCCGGACGGAAAAGATTTATCGGTGTGAGATGGACCCGCGTTGGATTAGCTAGTTGGCAGGGTAACGGCCTACCAAGGCGACGATCCATAGCCGGCCTGAGAGGGTGAACGGCCACATTGGGACTGAGACGCGGCCCAAACTCCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGAGTGAAGAAGTATTTCGGTATGTAAAGCTCTATCAGCAGGGAAGAAAATGACGGTACCTGACTAAGAAGCCCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGGGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGTTAAGCAAGTCAGAAGTGAAAGGCTGGGGCTCAACCCCGGGACTACTTTTGAAACTGTTTTAACTAGAGTGCTGGAGAGGTAAGCGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCTTACTGGACAGTAACTGACGTTGAGGCTCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAATACTAGGTGTCGGGGAGCACAGCTCTTCGGTGCCGCCGCAAACGCAATAAGTATTCCACCTGGGGAGTACGTTCGCAAGAATGAAACTCAAAGGAATTGACGGGGACCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGAAGCAACGCGAAGAACCTTACCAAGTCTTGACATCCCGATGACCGGTCTGTAATGGGACCTTCTCTTCGGAGCATCGGTGACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCATGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAACCCTTATCTTCAGTAGCCAGCATTTAGGATGGGCACTCTGGAGAGACTGCCAGGGATAACCTGGAGGAAGGTGGGGATGACGTCAAATCATCATGCCCCTTATGACTTGGGCTACACACGTGCTACAATGGCGTAAACAAAGGGAAGCGAGAGTGTGAGCTTAAGCAAATCTCAAAAATAACGTCTCAGTTCGGATTGTAGTCTGCAACTCGACTACATGAAGCTGGAATCGCTAGTAATCGCGAATCAGAATGTCGCGGTGAATACGCTCCCGGGTCTGTACACACCGCCCGTCAA,null,cbb7a848eb86d54e8fa409043dd86fb5
3503301,URS00003574C5,2014-05-29T13:51:05.000Z,RNACEN,D476529D07D5B243,67,AGTAGCTCAGCAGGATAGAGCAACGGCCTTCTAGGCCGTCGATCGGGAGTTCGAATCTCTACTGGGA,null,cbb7d67640fac697b297d188a41cddda
3503302,URS00003574C6,2014-05-29T13:51:05.000Z,RNACEN,2D202B06BC22572E,588,TGCAGTCGAACGAGAAAGTGGAGCAATCCATGAGTGCAGTGGCGTACGGGTGAGTAACACGTGACTAACCTACCCTCGAGTGGGGAATAACCTCGGGAAACCGGGGCTAATACCGCATAACATCTACGGATCAAATGCGTAAGCAGCTTGAGGAGGGGGTCGCGGCCGATCAGCTA

Data successfully saved to rna_100_records table
